# Basicness Classifier - TLN
### Authors: _Simone Multari_, _Mattia Mondino_, _Loris Signoretti_
Il seguente progetto mira a sviluppare un classificatore in grado di assegnare un valore di `BASICNESS` alle parole. Questa misura riflette il grado di complessità di una parola, rappresentando quanto una parola sia comune e di facile comprensione.

## Risorse Utilizzate

- **Dataset di Training e Valutazione**  
  Il [dataset](https://github.com/federicotorrielli/stableKnowledge/tree/master/json_analyzer) fornisce una lista di parole annotate con il relativo synset e un valore di basicness: `middle` o `advanced`.

- **WordNet**  
  Utilizzato come risorsa lessicale per estrarre caratteristiche (features) linguistiche utili alla classificazione.

- **CMU Pronouncing Dictionary**  
  [CMU Pronouncing Dictionary](https://www.nltk.org/_modules/nltk/corpus/reader/cmudict.html) è stato impiegato per calcolare la difficoltà di pronuncia delle parole, basandosi sul numero e la complessità dei fonemi.


- **Corpus di Storie per Bambini**  
  [Dataset di storie](https://huggingface.co/datasets/lilithyu/kaggle-child-stories) per bambini, impiegato per analizzare la frequenza delle parole.


## Modello di Classificazione

Per il training è stato utilizzato un **Random Forest Classifier**, addestrato sulle caratteristiche estratte per prevedere il livello di basicness delle parole.

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import KFold, cross_val_score
import pandas as pd

import nltk
# nltk.download('cmudict')
# nltk.download('wordnet')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.corpus import cmudict
from nltk.corpus import wordnet as wn
from nltk.corpus import cmudict

from gensim.corpora import Dictionary
from basicness_f import read_dataset, pronunciation_difficulty, string_to_synset


pronouncing_dict = cmudict.dict()

In [43]:
filename = './data/half_cleaned_merged_fairy_tales_without_eos.txt'
with open(filename, 'r') as file:
    corpus = file.read()

dataset_df = read_dataset('./data/1.json')
dataset_df.head()

,synset,word,target
0,Synset('war.n.01'),war,1
1,Synset('war.n.01'),warfare,1
2,Synset('fiefdom.n.01'),fiefdom,0
3,Synset('bed.n.03'),bed,1
4,Synset('bed.n.03'),bottom,1


### Features extraction da wordnet:

In [55]:
dataset_df['word_length'] = dataset_df['word'].apply(lambda x: len(x))
dataset_df['pronunciation_difficulty'] = dataset_df['word'].apply(pronunciation_difficulty)
dataset_df['synset_depth'] = dataset_df['synset'].apply(lambda x: string_to_synset(x).min_depth())
dataset_df['synset_max_depth'] = dataset_df['synset'].apply(lambda x: string_to_synset(x).max_depth())
dataset_df['synset_num_hypernyms'] = dataset_df['synset'].apply(lambda x: len(string_to_synset(x).hypernyms()))
dataset_df['synset_num_hyponyms'] = dataset_df['synset'].apply(lambda x: len(string_to_synset(x).hyponyms()))
dataset_df['synset_num_lemmas'] = dataset_df['synset'].apply(lambda x: len(string_to_synset(x).lemmas()))
dataset_df['word_num_senses'] = dataset_df['word'].apply(lambda x: len(wn.synsets(x)))
dataset_df['synset_gloss_length'] = dataset_df['synset'].apply(lambda x: len(string_to_synset(x).definition()))
dataset_df['synset_examples_length'] = dataset_df['synset'].apply(lambda x: sum(len(example) for example in string_to_synset(x).examples()))
dataset_df.head()

,synset,word,target,word_length,pronunciation_difficulty,synset_depth,synset_max_depth,synset_num_hypernyms,synset_num_hyponyms,synset_num_lemmas,word_num_senses,synset_gloss_length,synset_examples_length,dfs
0,Synset('war.n.01'),war,1,3,4,6,7,1,9,2,5,45,42,1.656566
1,Synset('war.n.01'),warfare,1,7,8,6,7,1,9,2,2,45,42,0.161616
2,Synset('fiefdom.n.01'),fiefdom,0,7,8,6,6,1,0,1,2,38,0,0.000000
3,Synset('bed.n.03'),bed,1,3,4,5,5,1,4,2,13,53,41,2.000000
4,Synset('bed.n.03'),bottom,1,6,7,5,5,1,4,2,12,53,41,0.949495


### Features extraction dal corpus di storie:

In [45]:
documents = corpus.split("\n\n")
documents = [word_tokenize(doc.lower()) for doc in documents]
# remove stopwords and non-alphabetic characters
stop_words = set(stopwords.words('english'))
documents = [[word for word in doc if word not in stop_words and word.isalpha()] for doc in documents]
# lemmatize
wnl = nltk.WordNetLemmatizer()
documents = [[wnl.lemmatize(word) for word in doc] for doc in documents]

documents[0][:10]

['happy',
 'prince',
 'high',
 'city',
 'tall',
 'column',
 'stood',
 'statue',
 'happy',
 'prince']

In [46]:
dictionary = Dictionary(documents)
dfs = dictionary.dfs

dfs.get(dictionary.token2id['prince']) / dictionary.num_docs

0.18383838383838383

In [64]:
dataset_df["dfs_fairy_tales"] = dataset_df["word"].apply(lambda word: (100*(dictionary.dfs[dictionary.token2id[word]] if word in dictionary.token2id else 0) / dictionary.num_docs))
dataset_df.head()

,synset,word,target,word_length,pronunciation_difficulty,synset_depth,synset_max_depth,synset_num_hypernyms,synset_num_hyponyms,synset_num_lemmas,word_num_senses,synset_gloss_length,synset_examples_length,dfs,dfs_fairy_tales
0,Synset('war.n.01'),war,1,3,4,6,7,1,9,2,5,45,42,16.565657,16.565657
1,Synset('war.n.01'),warfare,1,7,8,6,7,1,9,2,2,45,42,1.616162,1.616162
2,Synset('fiefdom.n.01'),fiefdom,0,7,8,6,6,1,0,1,2,38,0,0.000000,0.000000
3,Synset('bed.n.03'),bed,1,3,4,5,5,1,4,2,13,53,41,20.000000,20.000000
4,Synset('bed.n.03'),bottom,1,6,7,5,5,1,4,2,12,53,41,9.494949,9.494949


### plot valori medi delle features per le due classi

In [65]:
features = ["word_length", "pronunciation_difficulty", "synset_depth", "synset_max_depth", "synset_num_hypernyms", "synset_num_hyponyms", "synset_num_lemmas", "word_num_senses", "synset_gloss_length", "synset_examples_length", "dfs_fairy_tales"]
mean_features_df = pd.DataFrame(columns=['basic', 'advanced'])
for feature in features:
    mean_features_df.loc[feature] = [round(dataset_df[dataset_df['target'] == 1][feature].mean(), 2), round(dataset_df[dataset_df['target'] == 0][feature].mean(),2)]
mean_features_df

,basic,advanced
word_length,7.34,9.31
pronunciation_difficulty,8.07,9.85
synset_depth,6.27,7.37
synset_max_depth,6.61,7.59
synset_num_hypernyms,1.03,1.00
synset_num_hyponyms,9.06,3.73
synset_num_lemmas,4.10,3.95
word_num_senses,8.29,3.79
synset_gloss_length,57.90,62.06
synset_examples_length,54.72,25.18


### Valutazione del modello

In [66]:
X = dataset_df[features].values
y = dataset_df["target"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9193548387096774
Precision: 0.9316770186335404
Recall: 0.9433962264150944
F1: 0.9375
              precision    recall  f1-score   support

           0       0.90      0.88      0.89        89
           1       0.93      0.94      0.94       159

    accuracy                           0.92       248
   macro avg       0.91      0.91      0.91       248
weighted avg       0.92      0.92      0.92       248



### Valutazione con Cross Validation

In [67]:
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf, scoring="accuracy")

print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

Accuracy: 0.90 (+/- 0.05)
